# Disaster Tweet Classification: BERT Transformer Fine-Tuning

**Architecture:** BERT Base Uncased (`bert-base-uncased`, 110M Parameters)  
**Objective:** End-to-end transformer fine-tuning with HuggingFace, PyTorch AMP (Mixed Precision), AdamW optimizer with Linear Warmup, Class-Weighted CrossEntropy Loss, model serialization, and comprehensive disaster classification diagnostics across 10 classes.

---
### Notebook Structure
1. **Dataset Ingestion & Environment Setup**
2. **BERT Tokenization & PyTorch Dataset Preparation**
3. **Pretrained Transformer Model Setup & Class-Weighted Loss Function**
4. **Fine-Tuning Engine (FP16 Mixed Precision, Warmup Scheduler, Early Stopping)**
5. **Model Checkpointing (`save_pretrained` for Model and Tokenizer)**
6. **Diagnostic Evaluation Suite (Loss/F1 Curves, Confusion Matrices, Per-Class Bar Charts, Imbalance Robustness)**

In [ ]:
import os
import sys
import time
import random
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

# Setup directories
DATA_DIR = Path("dataset")
RESULTS_DIR = Path("results/05_transformer_bert")
MODELS_DIR = RESULTS_DIR / "saved_model"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Device & Reproducibility
SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Utilizing compute device: {device}")

train_path = DATA_DIR / "train_clean.parquet"
val_path = DATA_DIR / "validation_clean.parquet"
test_path = DATA_DIR / "test_clean.parquet"

# Sourcing dataset from Kaggle or Google Drive
KAGGLE_INPUT_DIR = Path("/kaggle/input/humaid-disaster-tweets-parquet")
if KAGGLE_INPUT_DIR.exists():
    print("[+] Sourcing dataset from Kaggle dataset input...")
    for split in ["train", "validation", "test"]:
        p_clean = KAGGLE_INPUT_DIR / f"{split}_clean.parquet"
        p_raw = KAGGLE_INPUT_DIR / f"{split}.parquet"
        target_p = DATA_DIR / f"{split}_clean.parquet"
        if not target_p.exists():
            if p_clean.exists():
                pd.read_parquet(p_clean).to_parquet(target_p)
            elif p_raw.exists():
                pd.read_parquet(p_raw).to_parquet(target_p)

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    raw_train = DATA_DIR / "train.parquet"
    raw_val = DATA_DIR / "validation.parquet"
    raw_test = DATA_DIR / "test.parquet"
    if not (raw_train.exists() and raw_val.exists() and raw_test.exists()):
        print("[+] Downloading HumAID dataset from Google Drive...")
        import gdown
        GDRIVE_URL = "https://drive.google.com/drive/folders/1pyMBc4SFc-sQvfmReiywPoN5cQbMpQBR?usp=drive_link"
        gdown.download_folder(url=GDRIVE_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)

train_file = train_path if train_path.exists() else DATA_DIR / "train.parquet"
val_file = val_path if val_path.exists() else DATA_DIR / "validation.parquet"
test_file = test_path if test_path.exists() else DATA_DIR / "test.parquet"

train_df = pd.read_parquet(train_file)
val_df = pd.read_parquet(val_file)
test_df = pd.read_parquet(test_file)

text_col = "clean_text" if "clean_text" in train_df.columns else "tweet_text"
print(f"[+] Loaded splits using column '{text_col}':")
print(f"    Train: {len(train_df):,} samples | Val: {len(val_df):,} samples | Test: {len(test_df):,} samples")

## 1. BERT Tokenization & Dataset Preparation
We initialize `bert-base-uncased` tokenizer with max sequence length 128 and construct PyTorch data loaders.

In [ ]:
MODEL_CHECKPOINT = "bert-base-uncased"
print(f"[+] Loading Tokenizer for {MODEL_CHECKPOINT}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# Encode labels
class_names = sorted(train_df["class_label"].unique())
label2idx = {name: i for i, name in enumerate(class_names)}
idx2label = {i: name for i, name in enumerate(class_names)}

y_train = train_df["class_label"].map(label2idx).values
y_val = val_df["class_label"].map(label2idx).values
y_test = test_df["class_label"].map(label2idx).values

MAX_LEN = 128
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = TransformerDataset(train_df[text_col], y_train, tokenizer)
val_dataset = TransformerDataset(val_df[text_col], y_val, tokenizer)
test_dataset = TransformerDataset(test_df[text_col], y_test, tokenizer)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"[+] Prepared PyTorch DataLoaders with batch size = {BATCH_SIZE}.")

## 2. Model Initialization & Class-Weighted Loss
We instantiate `AutoModelForSequenceClassification` with 10 output logits, class-balanced Cross-Entropy Loss, and AdamW optimizer.

In [ ]:
print(f"[+] Initializing Pretrained Model: {MODEL_CHECKPOINT}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(class_names)
).to(device)

# Compute Class Weights for Loss
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

EPOCHS = 4
LEARNING_RATE = 2e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler = GradScaler()
print(f"[+] Configured AdamW with lr={LEARNING_RATE}, total_steps={total_steps}, warmup_steps={warmup_steps}.")

## 3. Fine-Tuning Loop with FP16 Mixed Precision & Early Stopping

In [ ]:
history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}
best_val_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    
    # Training Phase
    model.train()
    total_train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_train_loss += loss.item() * len(labels)
        
    train_loss = total_train_loss / len(train_dataset)
    
    # Validation Phase
    model.eval()
    total_val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            with autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = criterion(logits, labels)
                
            total_val_loss += loss.item() * len(labels)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
    val_loss = total_val_loss / len(val_dataset)
    val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_macro_f1"].append(val_f1)
    
    elapsed = time.time() - start_time
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} [{elapsed:.1f}s] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro F1: {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        print(f"    --> Best validation Macro F1 improved to {val_f1:.4f}. Saving model & tokenizer...")
        model.save_pretrained(MODELS_DIR)
        tokenizer.save_pretrained(MODELS_DIR)

## 4. Final Evaluation on Test Split

In [ ]:
print("[+] Loading best model checkpoint for final Test set evaluation...")
model = AutoModelForSequenceClassification.from_pretrained(MODELS_DIR).to(device)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_labels.extend(batch["labels"].numpy())

test_acc = accuracy_score(test_labels, test_preds)
test_f1_macro = f1_score(test_labels, test_preds, average='macro', zero_division=0)
test_f1_weighted = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
test_prec_macro = precision_score(test_labels, test_preds, average='macro', zero_division=0)
test_rec_macro = recall_score(test_labels, test_preds, average='macro', zero_division=0)

metrics_summary = pd.DataFrame([{
    "Model": "BERT (bert-base-uncased)",
    "Test Accuracy": test_acc,
    "Test Macro F1": test_f1_macro,
    "Test Weighted F1": test_f1_weighted,
    "Test Macro Precision": test_prec_macro,
    "Test Macro Recall": test_rec_macro,
}])
metrics_summary.to_csv(RESULTS_DIR / "metrics.csv", index=False)

print("\n=== Final Test Evaluation Metrics ===")
print(metrics_summary.to_string(index=False))

## 5. Visualizations & Diagnostic Reports

In [ ]:
# 1. Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), dpi=120)
ax1.plot(history["train_loss"], label="Train Loss", marker='o')
ax1.plot(history["val_loss"], label="Val Loss", marker='s')
ax1.set_title("Loss Trajectory", fontsize=12, fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross Entropy Loss")
ax1.legend()

ax2.plot(history["val_macro_f1"], label="Val Macro F1", color="#27ae60", marker='^')
ax2.set_title("Validation Macro F1-Score Progression", fontsize=12, fontweight='bold')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Macro F1")
ax2.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", bbox_inches='tight')
plt.show()

# 2. Classification Report
report_str = classification_report(test_labels, test_preds, target_names=class_names, digits=4)
with open(RESULTS_DIR / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write("=== Classification Report: BERT (bert-base-uncased) ===\n\n")
    f.write(report_str)
print(report_str)

# 3. Dual Confusion Matrices
cm_raw = confusion_matrix(test_labels, test_preds)
cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8), dpi=120)
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax1)
ax1.set_title("Raw Confusion Matrix: BERT", fontsize=12, fontweight='bold')
ax1.set_xlabel("Predicted Label")
ax1.set_ylabel("True Label")
ax1.tick_params(axis='x', rotation=45)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax2)
ax2.set_title("Normalized Confusion Matrix: BERT", fontsize=12, fontweight='bold')
ax2.set_xlabel("Predicted Label")
ax2.set_ylabel("True Label")
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrices.png", bbox_inches='tight')
plt.show()

# 4. Per-Class Metrics Bar Chart
report_dict = classification_report(test_labels, test_preds, target_names=class_names, output_dict=True)
per_class_df = pd.DataFrame([
    {
        "Class": cls,
        "Precision": report_dict[cls]["precision"],
        "Recall": report_dict[cls]["recall"],
        "F1-Score": report_dict[cls]["f1-score"],
        "Support": report_dict[cls]["support"]
    }
    for cls in class_names
])
per_class_df.to_csv(RESULTS_DIR / "per_class_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(14, 7), dpi=120)
x = np.arange(len(class_names))
width = 0.25

ax.barh(x - width, per_class_df["Precision"], width, label="Precision", color="#3498db")
ax.barh(x, per_class_df["Recall"], width, label="Recall", color="#2ecc71")
ax.barh(x + width, per_class_df["F1-Score"], width, label="F1-Score", color="#e74c3c")

ax.set_yticks(x)
ax.set_yticklabels(class_names, fontsize=10)
ax.set_xlabel("Score", fontsize=11, fontweight='bold')
ax.set_title("Per-Class Performance Metrics: BERT", fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "per_class_metrics.png", bbox_inches='tight')
plt.show()

# 5. Imbalance Robustness Correlation
fig, ax = plt.subplots(figsize=(9, 5), dpi=120)
sns.regplot(data=per_class_df, x="Support", y="F1-Score", scatter_kws={'s': 60, 'color': '#8e44ad'}, line_kws={'color': '#2980b9'}, ax=ax)
ax.set_title("Class Support vs. F1-Score: BERT", fontsize=12, fontweight='bold')
ax.set_xlabel("Test Class Support (Number of Samples)", fontsize=11)
ax.set_ylabel("Class F1-Score", fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "imbalance_robustness_correlation.png", bbox_inches='tight')
plt.show()

print("[+] All artifacts and plots successfully saved to:", RESULTS_DIR.resolve())